# 自动断开测试（CPU 运行时即可，不用 GPU）

这里用的“布置自动断开”和“最后：自动断开”两格，与两个正式 notebook 里的**一字不差**（单元测试核对过）。断开以后运行时就没了，所以两种情况要各跑一次：

1. `MODE = "finish"`：全部运行。约 1 分钟后应当自动断开（页面右上角显示未连接，或弹出“运行时已断开”之类的提示）。
2. `MODE = "error"`：重新连接，改成 `"error"`，再全部运行。应当先打印测试报错、写出报错文件，再自动断开。
3. （可选）测试停止按钮：`MODE = "finish"`，在倒计时的那一格点停止。这时**不应**断开，后面几格也不会再跑，运行时保持连接；测完手动断开。

每次跑完到 Google Drive 里看 `qwen38_efficientthink/run_logs/`：
- 每次运行都会有一个 `disconnect-test-…-marker.txt`：断开前写的，在 Drive 上看得到，说明“先把 Drive 写完再断开”生效了；
- `MODE = "error"` 那次还会多一个 `disconnect-test-…-error.txt`，里面是测试报错。

最后可以在“代码执行程序 → 管理会话”里确认没有还在运行的会话。


In [ ]:
MODE = "finish"              # "finish"：正常跑完后断开；"error"：中途出错后断开
DRIVE_DIR = "/content/drive/MyDrive/qwen38_efficientthink"
AUTO_DISCONNECT = True
NOTEBOOK_TAG = "disconnect-test"

from pathlib import Path
from google.colab import drive as _drive
if not Path("/content/drive/MyDrive").exists():
    _drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print("Drive 已挂载；MODE =", MODE)


In [ ]:
# 夜里无人看着时用：AUTO_DISCONNECT = True 时，全部跑完或中途出错，都先把 Drive 写完，再断开运行时（停止计费）。
# 出错时报错另存一份到 DRIVE_DIR/run_logs/，断开后也能看。手动中断（停止按钮）不会触发断开。
import datetime as _dt
import traceback as _traceback
from pathlib import Path as _Path

RUN_LOG_DIR = _Path(DRIVE_DIR) / "run_logs"


def _release_runtime(reason):
    print(f"\n{reason}：先把 Drive 写完，再断开运行时（AUTO_DISCONNECT = True）。", flush=True)
    try:
        from google.colab import drive as _d
        _d.flush_and_unmount()
    except Exception as _e:                      # noqa: BLE001
        print("Drive 同步：", _e, flush=True)
    from google.colab import runtime as _rt
    _rt.unassign()


def _on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    try:
        RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
        _path = RUN_LOG_DIR / f"{NOTEBOOK_TAG}-{_dt.datetime.now():%Y%m%d-%H%M%S}-error.txt"
        _path.write_text("".join(_traceback.format_exception(etype, evalue, tb)), encoding="utf-8")
        print("报错已存到", _path, flush=True)
    except Exception as _e:                      # noqa: BLE001
        print("报错没能存下：", _e, flush=True)
    _release_runtime("出错了")


if AUTO_DISCONNECT:
    from IPython import get_ipython
    get_ipython().set_custom_exc((Exception,), _on_error)
    print("AUTO_DISCONNECT = True：全部跑完或出错后会自动断开运行时；出错时报错存在", RUN_LOG_DIR)
else:
    print("AUTO_DISCONNECT = False：跑完后运行时保持连接（继续计费），记得手动断开。")


In [ ]:
import datetime, time
_marker = Path(DRIVE_DIR) / "run_logs" / f"disconnect-test-{datetime.datetime.now():%Y%m%d-%H%M%S}-marker.txt"
_marker.parent.mkdir(parents=True, exist_ok=True)
_marker.write_text(f"写于 {datetime.datetime.now()}，MODE = {MODE}\n", encoding="utf-8")
print("已写出", _marker)
for _left in range(30, 0, -5):
    print(f"模拟计算中，还有 {_left} 秒（想测停止按钮就在这时点停止）", flush=True)
    time.sleep(5)


In [ ]:
if MODE == "error":
    raise RuntimeError("这是测试用的报错：看到它之后应当自动断开")
print("没有出错，继续到最后一格")


## 最后：自动断开

`AUTO_DISCONNECT = True` 时，这一格先把 Drive 写完，再断开运行时；结果都已经在 Drive 上。`False` 时什么也不做，记得手动“断开连接并删除运行时”。


In [ ]:
if AUTO_DISCONNECT:
    _release_runtime("全部跑完")
